In [1]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── 数据准备 ─────────────────────────────────────
transform = transforms.ToTensor()
full_train_set = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform)

# 划分训练集和验证集（9:1），用于 Early Stopping
train_size = int(0.9 * len(full_train_set))
val_size = len(full_train_set) - train_size
train_set, val_set = random_split(full_train_set, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

# ── 1. 初始化：网络结构 + He 初始化 ──────────────
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 10),
).to(device)

# He 初始化（打破对称性 + 控制信号尺度）
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)

# ── 3. 交叉熵损失函数 ─────────────────────────────
loss_fn = nn.CrossEntropyLoss()   # 多分类任务

# ── 5. 优化器 + 学习率调度 + 权重衰减 ───────────────
epochs = 30
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)  # AdamW + 权重衰减

total_steps = epochs * len(train_loader)
warmup_steps = int(0.1 * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)   # Warmup + Cosine 调度

# ── 6. 训练循环 + Early Stopping（在验证集上监控）────
best_val_loss = float('inf')
patience, no_improve = 6, 0
best_model_path = 'best_model.pt'

for epoch in range(epochs):
    # 训练阶段
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        output = model(xb)                    # 前向传播
        loss = loss_fn(output, yb)            # 计算损失
        loss.backward()                       # 反向传播
        optimizer.step()                      # 更新参数
        scheduler.step()                      # 学习率调度

    # 验证阶段（使用 val_loader，而不是 test_loader）
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            val_loss += loss_fn(out, yb).item() * yb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)
    val_loss /= total
    val_acc = correct / total

    print(f'Epoch {epoch+1:2d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    # Early Stopping 判断
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

# 加载验证集上表现最好的模型
model.load_state_dict(torch.load(best_model_path))

# ── 最终在测试集上评估（与验证集严格区分）────────────
model.eval()
test_loss, correct, total = 0.0, 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        test_loss += loss_fn(out, yb).item() * yb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
test_loss /= total
test_acc = correct / total
print(f'Final Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}')

/Users/qx/ai/deep_learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch  1 | val_loss=0.2324 | val_acc=0.9378
Epoch  2 | val_loss=0.1341 | val_acc=0.9610
Epoch  3 | val_loss=0.1103 | val_acc=0.9678
Epoch  4 | val_loss=0.0866 | val_acc=0.9715
Epoch  5 | val_loss=0.0821 | val_acc=0.9758
Epoch  6 | val_loss=0.0759 | val_acc=0.9758
Epoch  7 | val_loss=0.0706 | val_acc=0.9777
Epoch  8 | val_loss=0.0734 | val_acc=0.9788
Epoch  9 | val_loss=0.0694 | val_acc=0.9792
Epoch 10 | val_loss=0.0621 | val_acc=0.9812
Epoch 11 | val_loss=0.0670 | val_acc=0.9808
Epoch 12 | val_loss=0.0668 | val_acc=0.9800
Epoch 13 | val_loss=0.0666 | val_acc=0.9810
Epoch 14 | val_loss=0.0661 | val_acc=0.9822
Epoch 15 | val_loss=0.0619 | val_acc=0.9833
Epoch 16 | val_loss=0.0624 | val_acc=0.9823
Epoch 17 | val_loss=0.0617 | val_acc=0.9840
Epoch 18 | val_loss=0.0621 | val_acc=0.9833
Epoch 19 | val_loss=0.0641 | val_acc=0.9832
Epoch 20 | val_loss=0.0621 | val_acc=0.9830
Epoch 21 | val_loss=0.0620 | val_acc=0.9842
Epoch 22 | val_loss=0.0597 | val_acc=0.9835
Epoch 23 | val_loss=0.0606 | val